## App Review Extraction hek and viactiv

This notebook extracts user reviews from the Google Play Store and the Apple App Store for selected health insurance applications. The goal is to create a unified dataset of user feedback that can be used for further analysis and manual labeling.

### Steps performed in this notebook:

1. **Data Extraction**
   - Reviews are collected from:
     - Google Play Store using the `google_play_scraper`
     - Apple App Store using the `app_store_scraper` with an RSS-based fallback
   - For Google Play, multiple country and query configurations are used to maximize coverage.
   - For Apple, reviews are retrieved per country due to API limitations.

2. **Data Consolidation**
   - Reviews from both platforms are combined into a single dataframe.
   - A consistent schema is applied across all reviews, including fields such as:
     - review ID
     - source (store)
     - app name and identifier
     - review date
     - rating
     - review text

3. **Data Cleaning and Standardization**
   - Review text is cleaned (e.g., removing unnecessary whitespace).
   - Dates are converted into a consistent datetime format.
   - Duplicate reviews are removed based on unique identifiers.

4. **Timeframe Filtering**
   - Reviews are filtered to a defined timeframe:
     - Start date: `START_DATE`
     - End date: `END_DATE`
   - This ensures that only recent and relevant feedback is included.

5. **Data Storage**
   - The final dataset is saved as a CSV file in the `data/raw` directory.
   - This file serves as the input for subsequent steps, such as manual labeling and model development.

### Output

The result of this notebook is a structured dataset of app store reviews that can be used for:
- exploratory analysis
- manual labeling
- training and evaluation of NLP models

In [ ]:
from __future__ import annotations
import re
import hashlib
import requests
import time  # <--- FIXED: Added missing import
import numpy as np
import pandas as pd
from datetime import datetime
from pathlib import Path
from typing import Optional, Any, Union
from dateutil import parser

# Scrapers
from google_play_scraper import Sort, reviews_all
from app_store_scraper import AppStore

In [ ]:
# Setup Folders
DATA_DIR = Path("../data")
RAW_DIR = DATA_DIR / "raw"
PROCESSED_DIR = DATA_DIR / "processed"
LABELING_DIR = DATA_DIR / "labeling"

for folder in [RAW_DIR, PROCESSED_DIR, LABELING_DIR]:
    folder.mkdir(parents=True, exist_ok=True)

## Configuration

In [ ]:
START_DATE = "2023-12-01"
END_DATE = "2026-05-24"

# Google Play needs both LANG and COUNTRY to find all reviews
COUNTRIES = ["de", "fr", "pl", "gb", "us"] 
LANGUAGES = ["de", "en", "fr", "pl"]

LABEL_SAMPLE_SIZE = 150
RANDOM_STATE = 42

APPS = [
    {
        "app_name": "HEK Service-App",
        "google_play_app_id": "de.hek.serviceapp",
        "apple_app_id_or_url": "1287511413",
    },
        {
        "app_name": "VIACTIV - Service",
        "google_play_app_id": "de.viactiv.meinservice",
        "apple_app_id_or_url": "1608541708",
    },
        {
        "app_name": "VIACTIV - ePA",
        "google_play_app_id": "de.viactiv.meinegesundheit.live",
        "apple_app_id_or_url": "1540247793",
    },
]

## Robust Extraction Functions

In [ ]:
# ------------------------------------------------------------
# Helper functions
# ------------------------------------------------------------

def _label(obj, default=None):
    """Apple RSS fields are often {'label': value}; safely extract them."""
    if isinstance(obj, dict):
        return obj.get("label", default)
    return default


def extract_app_id(app_id_or_url: str) -> str:
    """Accepts either numeric Apple app ID or apps.apple.com URL."""
    match = re.search(r"id(\d+)", str(app_id_or_url))
    return match.group(1) if match else str(app_id_or_url)


# ------------------------------------------------------------
# Apple App Store reviews via RSS
# ------------------------------------------------------------

def fetch_apple_reviews_rss(
    app_id: str,
    country: str,
    app_name: Optional[str] = None,
    max_pages: int = 10,
    sleep_seconds: float = 1.0,
    timeout: int = 15,
) -> pd.DataFrame:
    """
    Fetch App Store reviews from Apple's public RSS customerreviews feed.

    Notes:
    - Apple RSS returns up to around 50 entries per page.
    - Page 1 often includes app metadata as the first entry.
    - Reviews contain 'im:rating'; app metadata does not.
    """

    app_id = extract_app_id(app_id)
    country = country.lower().strip()

    headers = {
        "User-Agent": (
            "Mozilla/5.0 (Macintosh; Intel Mac OS X 10_15_7) "
            "AppleWebKit/537.36 (KHTML, like Gecko) "
            "Chrome/120.0.0.0 Safari/537.36"
        ),
        "Accept": "application/json,text/json,*/*",
        "Accept-Language": f"{country},en;q=0.9",
    }

    rows = []

    with requests.Session() as session:
        session.headers.update(headers)

        for page in range(1, max_pages + 1):
            url = (
                f"https://itunes.apple.com/{country}/rss/customerreviews/"
                f"page={page}/id={app_id}/sortby=mostrecent/json"
            )

            try:
                resp = session.get(url, timeout=timeout)

                if resp.status_code != 200 or not resp.text.strip():
                    break

                try:
                    data = resp.json()
                except ValueError:
                    break

                entries = data.get("feed", {}).get("entry", [])

                if not entries:
                    break

                if isinstance(entries, dict):
                    entries = [entries]

                review_count_on_page = 0

                for entry in entries:
                    rating = _label(entry.get("im:rating"))

                    # The metadata entry does not have a rating.
                    if rating is None:
                        continue

                    review_count_on_page += 1

                    rows.append({
                        "review_id": _label(entry.get("id")),
                        "source_store": "apple_app_store",
                        "app_name": app_name,
                        "app_identifier": app_id,
                        "review_date": _label(entry.get("updated")),
                        "rating": int(rating) if str(rating).isdigit() else None,
                        "review_title": _label(entry.get("title"), ""),
                        "review_text": _label(entry.get("content"), ""),
                        "review_version": _label(entry.get("im:version"), ""),
                        "author": _label(entry.get("author", {}).get("name"), ""),
                        "country": country,
                        "language": None,
                    })

                if review_count_on_page == 0 or len(entries) < 50:
                    break

                time.sleep(sleep_seconds)

            except requests.RequestException:
                break

    df = pd.DataFrame(rows)

    if not df.empty:
        df["review_date"] = pd.to_datetime(df["review_date"], utc=True, errors="coerce")

        df = df.drop_duplicates(
            subset=["source_store", "app_identifier", "review_id"],
            keep="first"
        ).reset_index(drop=True)

    return df


# ------------------------------------------------------------
# Google Play reviews
# ------------------------------------------------------------

def fetch_google_play_comprehensive(
    app_name: str,
    package_name: str,
    countries: list,
    languages: list,
    sleep_milliseconds: int = 50,
) -> pd.DataFrame:
    """
    Fetch Google Play reviews across multiple countries and languages.
    """

    rows = []

    for lang in languages:
        for country in countries:
            try:
                result = reviews_all(
                    package_name,
                    lang=lang,
                    country=country,
                    sort=Sort.NEWEST,
                    sleep_milliseconds=sleep_milliseconds,
                )

                for item in result:
                    rows.append({
                        "review_id": item.get("reviewId"),
                        "source_store": "google_play",
                        "app_name": app_name,
                        "app_identifier": package_name,
                        "review_date": item.get("at"),
                        "rating": item.get("score"),
                        "review_title": "",
                        "review_text": item.get("content"),
                        "review_version": item.get("reviewCreatedVersion"),
                        "author": item.get("userName"),
                        "country": country,
                        "language": lang,
                    })

            except Exception as e:
                print(
                    f"Google Play failed | app={app_name} | "
                    f"country={country} | language={lang} | error={e}"
                )
                continue

    df = pd.DataFrame(rows)

    if not df.empty:
        df["review_date"] = pd.to_datetime(df["review_date"], utc=True, errors="coerce")

        df = df.drop_duplicates(
            subset=["source_store", "app_identifier", "review_id"],
            keep="first"
        ).reset_index(drop=True)

    return df


# ------------------------------------------------------------
# Main extraction function
# ------------------------------------------------------------

def fetch_all_app_reviews(
    apps: list,
    countries: list,
    languages: list,
    apple_max_pages: int = 10,
) -> tuple[pd.DataFrame, pd.DataFrame]:
    """
    Fetches Google Play and Apple App Store reviews for all apps, countries,
    and languages, then returns:

    1. df_total: one combined review DataFrame
    2. review_counts: summary table with review counts
    """

    final_frames = []

    for app in apps:
        app_name = app["app_name"]

        # ----------------------------
        # Google Play
        # ----------------------------
        google_play_app_id = app.get("google_play_app_id")

        if google_play_app_id:
            print(f"Extracting Google Play: {app_name}")

            gp_df = fetch_google_play_comprehensive(
                app_name=app_name,
                package_name=google_play_app_id,
                countries=countries,
                languages=languages,
            )

            print(f"  Google Play reviews found: {len(gp_df)}")

            if not gp_df.empty:
                final_frames.append(gp_df)

        # ----------------------------
        # Apple App Store
        # ----------------------------
        apple_app_id_or_url = app.get("apple_app_id_or_url")

        if apple_app_id_or_url:
            print(f"Extracting Apple App Store: {app_name}")

            apple_country_frames = []

            for country in countries:
                apple_df = fetch_apple_reviews_rss(
                    app_id=apple_app_id_or_url,
                    country=country,
                    app_name=app_name,
                    max_pages=apple_max_pages,
                )

                print(f"  Apple reviews found in {country}: {len(apple_df)}")

                if not apple_df.empty:
                    apple_country_frames.append(apple_df)

            if apple_country_frames:
                apple_app_df = pd.concat(apple_country_frames, ignore_index=True)
                final_frames.append(apple_app_df)

                print(f"  Apple total for {app_name}: {len(apple_app_df)}")

    # ----------------------------
    # Combine everything
    # ----------------------------
    if final_frames:
        df_total = pd.concat(final_frames, ignore_index=True)
    else:
        df_total = pd.DataFrame(columns=[
            "review_id",
            "source_store",
            "app_name",
            "app_identifier",
            "review_date",
            "rating",
            "review_title",
            "review_text",
            "review_version",
            "author",
            "country",
            "language",
        ])

    # ----------------------------
    # Final cleanup
    # ----------------------------
    if not df_total.empty:
        df_total["review_date"] = pd.to_datetime(
            df_total["review_date"],
            utc=True,
            errors="coerce"
        ).dt.tz_localize(None)

        df_total["review_text"] = df_total["review_text"].fillna("").astype(str)
        df_total["review_title"] = df_total["review_title"].fillna("").astype(str)

        # Safer deduplication:
        # 1. Prefer review_id where available.
        # 2. Fall back to text/date/rating only if review_id is missing.
        with_review_id = df_total[df_total["review_id"].notna()].drop_duplicates(
            subset=[
                "source_store",
                "app_identifier",
                "country",
                "language",
                "review_id",
            ],
            keep="first"
        )

        without_review_id = df_total[df_total["review_id"].isna()].drop_duplicates(
            subset=[
                "source_store",
                "app_identifier",
                "country",
                "language",
                "review_date",
                "rating",
                "review_text",
            ],
            keep="first"
        )

        df_total = pd.concat(
            [with_review_id, without_review_id],
            ignore_index=True
        )

        df_total = df_total.sort_values(
            by=["source_store", "app_name", "review_date"],
            ascending=[True, True, False]
        ).reset_index(drop=True)

    # ----------------------------
    # Review count summary
    # ----------------------------
    if not df_total.empty:
        review_counts = (
            df_total
            .groupby(["source_store", "app_name"], dropna=False)
            .size()
            .reset_index(name="reviews_retrieved")
            .sort_values(["source_store", "app_name"])
            .reset_index(drop=True)
        )
    else:
        review_counts = pd.DataFrame(
            columns=["source_store", "app_name", "reviews_retrieved"]
        )

    print("\nReview counts by store and app:")
    print(review_counts)

    print("\nReview counts by store:")
    print(
        df_total
        .groupby("source_store", dropna=False)
        .size()
        .reset_index(name="reviews_retrieved")
        if not df_total.empty
        else pd.DataFrame(columns=["source_store", "reviews_retrieved"])
    )

    return df_total, review_counts

## Execution & Filtering

In [ ]:
df_total, review_counts = fetch_all_app_reviews(
    apps=APPS,
    countries=COUNTRIES,
    languages=LANGUAGES,
    apple_max_pages=20,
)

df_total

In [ ]:
df_total.source_store.value_counts()

In [ ]:
# Timeframe filter
df_processed = df_total[(df_total["review_date"] >= START_DATE) & (df_total["review_date"] <= END_DATE)].copy()

print(f"\nTotal Processed Reviews: {len(df_processed)}")
print(df_processed["source_store"].value_counts())

## Save

In [ ]:
df_processed.to_excel(RAW_DIR / "reviews_final_hek_viactiv.xlsx", index=False)
print(f"Extraction Complete. Total Unique Reviews: {len(df_processed)}")
display(df_processed.head())